### Библиотеки

In [0]:
import pandas as pd
import numpy as np
from datetime import date 
import arrow

from datetime import datetime
# from spark_utils.df_helper import dtts
import pandas as pd

### Даты

In [0]:
event_start = '2026-01-01'
event_end = (date.today() + pd.Timedelta(days=1)).strftime('%Y-%m-%d')

### Уровни

In [0]:
lvl_start = 0
lvl_end = 18000

###Оценка CF

#### Ⓜ️ Расчет монетизационного CF

In [0]:
# все игроки
test = spark.sql(f"""
with

src as (
  select
      lvl.user_id,
      lvl.level,
      lvl.complexity,
      try_cast(lvl.event_payload:moves_default as int) as s_moves,
      lvl.event_payload:['goals_layer_move_unfinished.0'] as glmu0
  from bronze.levels_mym_amp lvl
  where
      lvl.partition_date between date('{event_start}') and date('{event_end}')
      and lvl.chain not like 'WL_%'
      and lvl.event_payload:goals_layer_total_unfinished is null
      and (try_cast(lvl.event_payload:purchase_additional_moves as int) + try_cast(lvl.event_payload:boosts_plus5moves as int)) = 1
      AND TRY_CAST(lvl.event_payload:level AS INT)
        = TRY_CAST(regexp_extract(lvl.event_payload:balance_id, 'st([0-9]+)', 1) AS INT)
      and lvl.level between {lvl_start} and {lvl_end}
      and cast(lvl.event_payload:at_drop_tails_started as boolean) is not TRUE
),

md as (
  select
      level,
      s_moves,
      count(distinct user_id) as users
  from src
  group by level, s_moves
),

pick as (
  select level, s_moves
  from (
    select
        level,
        s_moves,
        users,
        row_number() over(partition by level order by users desc, s_moves desc) as rn
    from md
  ) t
  where rn = 1
),

att_info as (
  select
      s.user_id,
      s.level,
      s.complexity,
      try_cast(
        split_part(
          split_part(
            s.glmu0,
            concat('{{', s.s_moves, ': '),
            2
          ),
          '}}',
          1
        ) as int
      ) as layers_on_add_moves
  from src s
  inner join pick p
    on p.level = s.level and p.s_moves = s.s_moves
)

select
    al.level,
    count(al.user_id) as users,
    round(percentile(layers_on_add_moves, 0.8), 0) as p80
from att_info al
where layers_on_add_moves is not null
group by 1
order by al.level

""")

test.display()
test.write.mode('overwrite').option("overwriteSchema", "true").saveAsTable('game_data_prod.analytics_voki.dk_mym_coridors_cf_p80_ML_churn_2026')

In [0]:
%sql
select * from game_data_prod.analytics_voki.dk_mym_coridors_cf_p80_2001_4000_SKO

####Исходы на попытках

In [0]:
df = spark.sql(f"""
with

payers as (
  select
      distinct pa.user_id
  from bronze.levels_mym_amp pa
  where 1=1
      and try_cast(pa.user_payload:payments as int) > 0
      and pa.partition_date between date('{event_start}') and date('{event_end}')
),

src as (
  select
      ae.user_id,
      ae.level,
      ae.attempt,
      ae.client_time,
      ae.partition_date,
      ae.reason,
      try_cast(ae.event_payload:moves_default as int) as moves_default,
      round(
        coalesce(
          try_cast(
            split_part(
              split_part(
                ae.event_payload:['goals_layer_move_unfinished.0'],
                concat('{{', try_cast(ae.event_payload:moves_default as int), ': '),
                2
              ),
              '}}',
              1
            ) as int
          ),
          0
        ),
        0
      ) as layer_last_move,
      case
        when try_cast(ae.event_payload:moves_left as int) < 0 then 0
        else try_cast(ae.event_payload:moves_left as int)
      end as moves_left
  from bronze.levels_mym_amp ae
    inner join 
        payers pa
        on ae.user_id = pa.user_id
  where 1=1
      and ae.partition_date between date('{event_start}') and date('{event_end}')
      and ae.chain not like 'WL_%'
        and try_cast(ae.event_payload:level as int)
            = try_cast(regexp_extract(ae.event_payload:balance_id, 'st([0-9]+)', 1) as int)
        and ae.level between {lvl_start} and {lvl_end}
),

outcome_info as (
  select
      us.user_id,
      us.level,
      us.attempt,
      us.client_time,
      us.partition_date,

      case
          when us.moves_left = 0 and us.layer_last_move > la.p80 then 1
          when us.reason = 'exit' then 1
          else 0
      end as FF,

      case
          when us.moves_left = 0 and us.layer_last_move > 0 and us.layer_last_move <= la.p80 then 1
          else 0
      end as CF,

      case
          when us.layer_last_move = 0 and us.moves_left between 0 and 2 then 1
          else 0
      end as CW,

      case
          when us.layer_last_move = 0 and us.moves_left >= 3 then 1
          else 0
      end as FW

  from src us
    inner join game_data_prod.analytics_voki.dk_mym_coridors_cf_p80_ML_churn_2026 la
    on la.level = us.level
)

select * from outcome_info


""")
# df.display()
df.write.mode('overwrite').option("overwriteSchema", "true").saveAsTable('game_data_prod.analytics_voki.dk_mym_outcomes_ML_churn_2026')


In [0]:
df.display()

In [0]:
df.select(F.max("partition_date")).show()

####Выгрузка данных их временных таблиц

In [0]:
%sql
select distinct level from game_data_prod.analytics_voki.dk_mym_outcomes_ML_churn_2026 order by level

In [0]:
%sql
-- select * from game_data_prod.analytics_voki.dk_mym_unfinished_layers_levels_absolut_w_SB
select * from game_data_prod.analytics_voki.dk_mym_unfinished_layers_levels_absolut_wo_SB_18
order by level, percent, CF_CW